# Авито: полный цикл с исходных данных

Здесь заново готовятся признаки, обучаются две модели CatBoost и создаётся `answer.csv`.
Готовых моделей конкурса и таблиц признаков в загружаемом комплекте нет.
Открытая языковая модель E5 используется как предобученный кодировщик, её веса не обучаются заново.

В Colab выберите **Runtime → Change runtime type → GPU**, затем выполняйте ячейки сверху вниз.
Нужен Python 3.12. Если версия среды изменилась, можно выбрать runtime 2026.07 [1].
Планируйте около 16 ГБ RAM и 20 ГБ диска с запасом. Полный запуск в Colab ещё не измерен.
Ресурсы и время сессии Colab ограничены [2].

После полного переобучения на CUDA новый CSV может отличаться от прежнего ответа с Apple MPS.
В конце сохраняются метрика этого запуска и контрольная сумма его CSV.

## 1. Подготовить рабочую папку и исходники

Архив содержит только код и параметры. Проверка SHA256 фиксирует именно эту версию алгоритма.
Для локального запуска положите ноутбук рядом с `avito_colab_source.zip`.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile

IN_COLAB = Path("/content").is_dir()
WORK = Path("/content/avito_from_scratch") if IN_COLAB else Path.cwd() / "run"
DEVICE = "cuda" if IN_COLAB else "auto"
DATASET_ARCHIVE = None  # Можно указать путь к самостоятельно загруженному dataset.zip.
SOURCE_URL = "https://raw.githubusercontent.com/revefffect/avito-contest/80ab9ca4ee3098ed83a3c63129926d363cb4a0d0/colab/avito_colab_source.zip"
SOURCE_SHA256 = "4e48828f105d3e416ab382d07c3765e3dd146b312c80ec7b1fc55496b59718fb"

if sys.version_info[:2] not in [(3, 11), (3, 12)]:
    raise RuntimeError("Выберите среду с Python 3.12, например Colab runtime 2026.07.")
WORK.mkdir(parents=True, exist_ok=True)
local_source = Path.cwd() / "avito_colab_source.zip"
source = WORK / "source.zip"
if local_source.is_file():
    shutil.copyfile(local_source, source)
else:
    urllib.request.urlretrieve(SOURCE_URL, source)
assert hashlib.sha256(source.read_bytes()).hexdigest() == SOURCE_SHA256
with zipfile.ZipFile(source) as bundle:
    for info in bundle.infolist():
        target = (WORK / info.filename).resolve()
        if not target.is_relative_to(WORK.resolve()) or info.filename.startswith("/"):
            raise ValueError("Недопустимый путь в архиве исходников")
        if Path(info.filename).suffix not in {".py", ".json", ".txt", ".toml"}:
            raise ValueError("В архиве должен быть только исходный код и параметры")
    bundle.extractall(WORK)
for name in ["reports", "artifacts", "data"]:
    (WORK / name).mkdir(exist_ok=True)
print("Рабочая папка:", WORK)
print("Свободный диск, ГБ:", round(shutil.disk_usage(WORK).free / 2**30, 1))


## 2. Установить библиотеки

Библиотеки устанавливаются в отдельное окружение. Версии numpy в основном ядре Colab
не влияют на дочерние процессы обучения, перезапуск ядра после установки не нужен.
PyTorch загружается из официального репозитория [3].

In [ ]:
venv = WORK / ".venv"
PYTHON = venv / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "venv", str(venv)], check=False)
pip_ready = (
    PYTHON.exists()
    and subprocess.run(
        [str(PYTHON), "-m", "pip", "--version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    ).returncode
    == 0
)
if not pip_ready:
    bootstrap = WORK / ".bootstrap"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--target",
            str(bootstrap),
            "virtualenv==20.34.0",
        ],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "virtualenv",
            "--no-download",
            str(venv),
        ],
        env={**os.environ, "PYTHONPATH": str(bootstrap)},
        check=True,
    )
subprocess.run(
    [
        str(PYTHON),
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        str(WORK / "requirements.txt"),
    ],
    check=True,
)
torch_command = [str(PYTHON), "-m", "pip", "install", "--quiet", "torch==2.9.1"]
if sys.platform.startswith("linux"):
    wheel_kind = "cu126" if DEVICE in ["auto", "cuda"] else "cpu"
    torch_command += ["--index-url", "https://download.pytorch.org/whl/" + wheel_kind]
subprocess.run(torch_command, check=True)
probe = (
    "import json, torch, platform"
    + chr(10)
    + "print(json.dumps({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"
)
environment = json.loads(subprocess.check_output([str(PYTHON), "-c", probe], text=True))
print(environment)
if DEVICE == "cuda" and not environment["cuda"]:
    raise RuntimeError(
        "GPU недоступен. Включите GPU в настройках Colab или явно выберите DEVICE = cpu."
    )
(WORK / "reports/environment.json").write_text(json.dumps(environment, indent=2))


def run_stage(stage):
    command = [
        str(PYTHON),
        "-u",
        str(WORK / "scripts/from_scratch.py"),
        "--stage",
        stage,
        "--device",
        DEVICE,
    ]
    if DATASET_ARCHIVE and stage == "download":
        command += ["--dataset-archive", str(DATASET_ARCHIVE)]
    subprocess.run(command, cwd=WORK, check=True)


## 3. Загрузить исходные данные

Скачиваются три исходные таблицы конкурса [4]. Контрольная сумма архива и число строк
проверяются автоматически. При недоступности Яндекс Диска загрузите `dataset.zip`
в Colab вручную и задайте его путь в `DATASET_ARCHIVE` первой ячейки.

In [ ]:
run_stage("download")


## 4. Подготовить разделение

Выделяются 3000 запросов fit, 1500 dev и 1500 audit. Модели обучаются на fit и dev,
так как их настройки уже зафиксированы прежними экспериментами [5]. Audit не участвует
в обучении или подборе параметров. Проверяемые пары удаляются из поведенческой истории.

Разделение по запросам не гарантирует полного разделения целевых объявлений.
Локальная метрика не является обещанием результата на скрытом тесте.

In [ ]:
run_stage("split")
print((WORK / "reports/validation_split.json").read_text())


## 5. Построить текстовый поиск

Словный TF-IDF и BM25 ищут точные слова, символьный TF-IDF помогает с вариантами написания.
Индексы строятся по заголовкам, параметрам и описаниям исходных объявлений.

In [ ]:
run_stage("lexical")


## 6. Рассчитать смысловые представления

Открытая E5 [6] кодирует каждое объявление и два варианта запроса, с фильтрами и без них.
Ревизия весов закреплена. Эмбеддинги рассчитываются заново, готовые векторы не скачиваются.
При прерывании этого этапа в той же рабочей папке можно повторить ячейку.

In [ ]:
run_stage("model")
run_stage("embeddings")


## 7. Собрать признаки для обучения

Кандидаты объединяются из текстового, смыслового и поведенческого поиска.
Для каждой пары рассчитываются сходство текстов, география, фильтры и статистики истории.
Идентификаторы используются для соединения таблиц, но не подаются в модели.

In [ ]:
run_stage("train_features")


## 8. Обучить две модели CatBoost

Первая модель использует прямую историю выбора объявления. Вторая исключает шесть таких
признаков, остальные статистики истории сохраняются. Глубина равна 6, скорость обучения
0,07, число деревьев 700 и 1000, seed 42. Обе модели обучаются заново на CPU.
Признак `category_match` исключён из-за сдвига категорий [5].

In [ ]:
config = json.loads((WORK / "configs/final.json").read_text())
print(json.dumps(config["models"], ensure_ascii=False, indent=2))
run_stage("train")


## 9. Проверить качество на audit

Для каждого запроса берётся доля найденных положительных объявлений, затем среднее по запросам:

$$
\operatorname{Recall@50} = \frac{1}{|Q|}\sum_{q\in Q}\frac{|T_{50}(q)\cap R(q)|}{|R(q)|}.
$$

Положительные объявления, не попавшие в кандидаты, остаются в знаменателе.
Проверяется заранее выбранная смесь с весами 0,5 и 0,5. По этому audit веса не меняются.

In [ ]:
run_stage("audit")
audit = json.loads((WORK / "reports/audit.json").read_text())
print(json.dumps(audit["results"], ensure_ascii=False, indent=2))


## 10. Рассчитать benchmark и сохранить ответ

Признаки benchmark строятся из исходных таблиц. Оценки моделей стандартизуются отдельно
внутри каждого запроса и усредняются:

$$
z_m(q,d)=\frac{s_m(q,d)-\mu_m(q)}{\sigma_m(q)},\qquad
s(q,d)=0.5z_1(q,d)+0.5z_2(q,d).
$$

При нулевом стандартном отклонении соответствующая оценка равна нулю.
Выбираются 50 объявлений с наибольшими оценками. Проверяются все запросы,
допустимость идентификаторов и отсутствие повторов.

In [ ]:
run_stage("benchmark")
run_stage("predict")
result = json.loads((WORK / "reports/reproducibility.json").read_text())
print("SHA256:", result["answer_sha256"])
print("Совпал с прежним CSV:", result["matches_previous_mps_answer"])


## 11. Скачать результат

Скачайте новый CSV и сохраните отчёты вместе с ним. Именно эта пара соответствует текущему
полному обучению. Архив отчётов не содержит больших признаков или моделей.

In [ ]:
answer = WORK / "answer.csv"
with zipfile.ZipFile(WORK / "run_reports.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted((WORK / "reports").rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(WORK))
if IN_COLAB:
    from google.colab import files

    files.download(str(answer))
    files.download(str(WORK / "run_reports.zip"))
else:
    print("Ответ:", answer)
    print("Отчёты:", WORK / "run_reports.zip")


## Источники

[1] [Версии среды Colab](https://research.google.com/colaboratory/runtime-version-faq.html).

[2] [Ресурсы и ограничения Colab](https://research.google.com/colaboratory/faq.html).

[3] [Установка PyTorch 2.9.1](https://pytorch.org/get-started/previous-versions/#v291).

[4] [Данные конкурса](https://disk.yandex.ru/d/sNhfo0YOjGtufg).

[5] [Исходное решение и проверенные эксперименты](https://github.com/revefffect/avito-contest/blob/main/evidence/INDEX.md).

[6] [E5, фиксированная ревизия](https://huggingface.co/intfloat/multilingual-e5-small/tree/614241f622f53c4eeff9890bdc4f31cfecc418b3).